<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/Howard2025_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Howard, Lohre & Mudde (2025): network-based investing

* An exploratory replication of the low-centrality factor and market-timing experiments described in *Causal Network Representations in Factor Investing*. The original notebook identifies the reference as *Intelligent Systems in Accounting, Finance and Management*, 32(1), e70001.

**Implementation status:** The code attempts to import DYNOTEARS from CausalNex and otherwise uses contemporaneous per-node Lasso. The saved output shows the **Lasso fallback**.

**Scope:** A manually chosen stock universe, monthly Yahoo Finance data requested for 2000–2022, and rolling 48-month network fits. The retained stock count depends on the downloaded data.


## Notebook guide

1. Dependencies and experiment settings
2. Prices, market-cap proxies and benchmark returns
3. Rolling network fits and centrality
4. Low-centrality portfolio experiment
5. Market-return prediction experiment
6. Diagnostic charts


## 1. Dependencies and experiment settings

* Request `pandas-datareader` and `causalnex`, import the libraries, and set a 48-month window, regularization values of 0.1, one lag and 100 observations of regression burn-in.

* The saved installation output reports a CausalNex compatibility failure.

* In the Lasso branch, only `LAMBDA_W` controls the penalty; `LAMBDA_A` and `N_LAGS` do not affect fitting.


In [ ]:
# A failed installation does not necessarily stop subsequent notebook statements.
# Check imports and package versions rather than relying on the final print message.
!pip install pandas-datareader causalnex -q

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas_datareader.data as web
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Experiment settings; not every parameter is used by the Lasso fallback.
WINDOW   = 48    # 4-year rolling window (Section 3.2)
LAMBDA_W = 0.1   # contemporaneous regularisation lambda_W (Section 3.2, Pamfil et al. 2020)
LAMBDA_A = 0.1   # lagged regularisation lambda_A (Section 4.1.2)
N_LAGS   = 1     # DYNOTEARS lag order p
GAMMA    = 3.0   # risk aversion for CER (Neely et al. 2014)
BURN_IN  = 100   # OOS burn-in months (Howard footnote 25)
START, END = '2000-01-01', '2022-12-31'

TICKERS = [
    'AAPL','MSFT','GOOGL','INTC','IBM','ORCL','QCOM','TXN','ADBE','CRM',
    'JPM','BAC','WFC','C','GS','MS','AXP','BLK','USB','COF',
    'AIG','MET','PRU','ALL','TRV',
    'JNJ','UNH','PFE','ABT','MRK','AMGN','GILD','BMY','MDT','ABBV',
    'AMZN','HD','MCD','NKE','SBUX','TGT','LOW','F','GM',
    'PG','KO','PEP','WMT','COST','MO','PM','CL','MDLZ',
    'GE','BA','HON','MMM','CAT','UNP','UPS','EMR','DE','GD','LMT','RTX',
    'XOM','CVX','COP','SLB','OXY','HAL','PSX','MPC','VLO',
    'DUK','SO','D','EXC','AEP','NEE','XEL',
    'AMT','SPG','PSA','T','VZ','LIN','DD','NEM',
]
print('Setup complete.')


## 2. Data and weighting proxies

* Download monthly adjusted prices, retain columns with less than 20% missing observations, then interpolate across the entire sample in both directions. Compute monthly log returns from the filled prices.

* **Availability caveat:** Full-sample filtering and interpolation use information beyond individual training windows. Interpolation has no gap-length limit and can extend endpoints. The static ticker list is not a reconstruction of historical index membership.

* For portfolio weights, multiply historical adjusted prices by **current** shares outstanding, with median-share imputation when retrieval fails. This is a rough size proxy with look-ahead and split-consistency concerns, not historical market capitalization.

* The market series is SPY log return less an approximate monthly risk-free rate (`TB3MS / 100 / 12`). Missing rates are filled with zero, and the entire rate series defaults to zero if FRED retrieval fails. This mixes log returns with a simple-rate approximation.


In [ ]:
raw = yf.download(TICKERS, start=START, end=END,
                  interval='1mo', auto_adjust=True, progress=False)
prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]
# Filter using the whole sample, then fill without a gap limit in both directions.
# This can use future observations and invent pre-listing or endpoint prices.
prices = prices.loc[:, prices.isnull().mean() < 0.20]
prices = prices.interpolate(method='linear', limit_direction='both').dropna(axis=1)
stocks = prices.columns.tolist()
stock_idx = {s: i for i, s in enumerate(stocks)}
N      = len(stocks)
log_rets = np.log(prices / prices.shift(1)).dropna()
print(f'{N} stocks | {log_rets.index[0].strftime("%Y-%m")} to {log_rets.index[-1].strftime("%Y-%m")} | {len(log_rets)} monthly observations')

# Market cap proxy: current shares outstanding x historical monthly price
# This is not point-in-time historical capitalization; current shares create look-ahead.
print('Downloading shares outstanding for VW weighting...')
shares_out = {}
for tk in stocks:
    try:
        fast = yf.Ticker(tk).fast_info
        so = None
        for attr in ['shares', 'shares_outstanding']:
            v = getattr(fast, attr, None)
            if v and v > 0:
                so = v; break
        shares_out[tk] = so if so else np.nan
    except:
        shares_out[tk] = np.nan
shares_ser = pd.Series(shares_out, dtype=float)
# Impute missing share counts with the cross-sectional median. If every
# retrieval failed, the median is also missing and this cannot produce valid weights.
shares_ser = shares_ser.fillna(shares_ser.median())
mcap_monthly = prices[stocks] * shares_ser.values   # (T, N) market cap proxy
mcap_np = mcap_monthly.values
print(f'Market cap proxy ready. Median share count: {shares_ser.median()/1e9:.1f} billion (current shares)')

# S&P 500 excess returns
spy_raw = yf.download('SPY', start=START, end=END, interval='1mo',
                       auto_adjust=True, progress=False)['Close'].squeeze()
spy_ret = np.log(spy_raw / spy_raw.shift(1)).dropna()
spy_ret.index = spy_ret.index.to_period('M')
try:
    tb = web.DataReader('TB3MS', 'fred', start=START, end=END)
    rf  = (tb['TB3MS'] / 100 / 12)
    rf.index = rf.index.to_period('M')
except Exception:
    print('FRED unavailable — using RF=0')
    rf = pd.Series(0.0, index=spy_ret.index)
spy_ex = spy_ret - rf.reindex(spy_ret.index).fillna(0)
print(f'S&P 500 excess returns: {len(spy_ex)} months')

## 3. Rolling network fits and centrality

* Standardize each stock within the preceding 48 monthly return observations and estimate one network per prediction month. The target month’s return is excluded from that network fit.

- **DYNOTEARS branch:** fit a dynamic structure, remove lag suffixes from node names, and retain the maximum absolute coefficient for each directed stock pair across returned edges. This is a maximum, not a sum over lags.
- **Lasso fallback:** regress each stock’s contemporaneous standardized returns on the other stocks. This uses no lagged predictors and imposes no acyclicity constraint. It is an association-network approximation.

* Symmetrize absolute coefficients, take the eigenvector associated with the largest eigenvalue, and normalize it to unit L2 norm. Store its mean across assets and the change in that mean.

* Mean normalized eigenvector centrality describes the spread of centrality across nodes; it is not literal edge density and is invariant to common positive scaling of adjacency weights. Tied leading eigenvalues can make rankings unstable. The saved output confirms the Lasso branch for that saved run.


In [ ]:
try:
    from causalnex.structure.dynotears import from_pandas_dynamic
    USE_DYNOTEARS = True
    print('CausalNex DYNOTEARS loaded OK')
except ImportError:
    from sklearn.linear_model import Lasso
    USE_DYNOTEARS = False
    print('WARNING: CausalNex not available — falling back to per-node Lasso.')
    print('         Results will differ from paper (no acyclicity constraint).')

T_total = len(log_rets)
n_wins  = T_total - WINDOW
log_np  = log_rets.values
all_centrality = np.zeros((n_wins, N))
pred_dates = log_rets.index[WINDOW:]

print(f'Fitting {n_wins} rolling {WINDOW}-month networks on {N} stocks...')
print(f'Method: {"DYNOTEARS" if USE_DYNOTEARS else "per-node Lasso (fallback)"}')
import time; t0 = time.time()

for t in range(n_wins):
    # Window t uses return rows t through t+WINDOW-1. The next return row
    # t+WINDOW is the prediction month associated with this centrality vector.
    W_slice = log_np[t : t + WINDOW]
    X_norm  = StandardScaler().fit_transform(W_slice)  # z-score per paper Sec 3.2

    if USE_DYNOTEARS:
        df_win = pd.DataFrame(X_norm, columns=stocks)
        sm = from_pandas_dynamic(df_win, p=N_LAGS,
                                 lambda_w=LAMBDA_W, lambda_a=LAMBDA_A)
        # Build adjacency: project all edges (lag 0 and lag 1) to N x N stock pairs
        B = np.zeros((N, N))
        for u, v, data in sm.edges.data(True):
            u_base = u.rsplit('_lag', 1)[0] if '_lag' in u else u
            v_base = v.rsplit('_lag', 1)[0] if '_lag' in v else v
            if u_base in stock_idx and v_base in stock_idx:
                i = stock_idx[u_base]
                j = stock_idx[v_base]
                w = abs(data.get('weight', 0))
                # Collapse lagged nodes to stock IDs and retain the largest absolute edge.
                # Potential self-lag edges are not explicitly removed after projection.
                B[j, i] = max(B[j, i], w)
    else:
        # Fallback: per-node Lasso (approximation — lacks acyclicity constraint)
        B = np.zeros((N, N))
        for i in range(N):
            # Fit simultaneous associations: target and predictors use the same months.
            # LAMBDA_A and N_LAGS are unused in this branch; cycles are allowed.
            y      = X_norm[:, i]
            X_rest = np.delete(X_norm, i, axis=1)
            lasso  = Lasso(alpha=LAMBDA_W, fit_intercept=True, max_iter=5000)
            lasso.fit(X_rest, y)
            cols = [j for j in range(N) if j != i]
            B[i, cols] = lasso.coef_

    A    = np.abs(B) + np.abs(B.T)
    # eigh returns eigenvalues in ascending order for symmetric A. Take the
    # last eigenvector, discard its sign and normalize. Degenerate eigenvalues
    # can yield nonunique centralities; mean centrality is not an edge-count statistic.
    ev, evec = np.linalg.eigh(A)
    cent = np.abs(evec[:, -1])
    norm = np.linalg.norm(cent)
    all_centrality[t] = cent / norm if norm > 0 else cent / N

    if (t + 1) % 20 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (t + 1) * (n_wins - t - 1)
        print(f'  {t+1}/{n_wins}  ({elapsed:.0f}s elapsed, ~{eta/60:.0f} min remaining)')

print(f'Done in {time.time()-t0:.0f}s')
avg_cent   = all_centrality.mean(axis=1)
# Difference consecutive fitted-window means. Assign each change to the
# later fit’s prediction month, whose return is not used in that fit.
delta_d    = np.diff(avg_cent)
dens_dates = pd.PeriodIndex(pred_dates[1:], freq='M')
print(f'avg_centrality: mean={avg_cent.mean():.4f}  std={avg_cent.std():.4f}')

## 4. Experiment 2: low-centrality portfolio

* Rank stocks using the preceding network fit.
* Go long the lowest-centrality group and short the highest-centrality group, with `floor(N/5)` stocks per leg. Normalize the size proxy within each leg, giving total long weight +1 and short weight −1.

* **Return convention:** The code subtracts weighted sums of stock log returns, then compounds `1 + factor_rets`. A weighted log-return spread is not an exact self-financing simple portfolio return, so the reported growth, drawdown and return statistics are approximations. No transaction or borrowing costs are modeled.

* **Alignment:** The code indexes prices and returns positionally even though returns lose the first price row. Its size-proxy row therefore needs an explicit date-alignment check; the final available forecast month is also omitted by `range(n_wins - 1)`.

* The t-test is an ordinary one-sample test with no serial-correlation adjustment. The market regression reports an annualized intercept and beta, without alpha significance. Printed paper values are historical reference text, not results independently verified in this tidy-up.


In [ ]:
# Size-proxy-weighted groups formed from prior-window centrality.
N_q = max(1, N // 5)
factor_rets, factor_dates = [], []

# Use all but the final available network forecast. The omitted last month
# is an implementation choice retained here, not required by the array lengths.
for t in range(n_wins - 1):
    cent      = all_centrality[t]
    ranked    = np.argsort(cent)          # ascending centrality
    long_idx  = ranked[:N_q]              # most peripheral (lowest centrality) -> long
    short_idx = ranked[-N_q:]             # most central (highest centrality) -> short
    r_month   = log_np[WINDOW + t]

    # Normalize size proxies within each leg, then aggregate stock log returns.
    # prices and log_rets do not share row positions after the first return is
    # dropped. This index is one price row earlier than the usual formation month
    # when only the initial return row is dropped; align by date before correcting it.
    mcap_t    = mcap_np[WINDOW + t - 1]  # positional size-proxy row; verify against return dates
    long_w    = mcap_t[long_idx];  long_w  = long_w  / long_w.sum()
    short_w   = mcap_t[short_idx]; short_w = short_w / short_w.sum()
    long_ret  = (r_month[long_idx]  * long_w).sum()
    short_ret = (r_month[short_idx] * short_w).sum()
    factor_rets.append(long_ret - short_ret)
    factor_dates.append(pred_dates[t])

factor_rets = np.array(factor_rets)
ann_ret = factor_rets.mean() * 12
ann_vol = factor_rets.std(ddof=1) * np.sqrt(12)
ann_sr  = ann_ret / ann_vol
# factor_rets is a weighted log-return spread. Treating it as simple returns
# for cumprod is an approximation, not exact portfolio wealth. Initial wealth
# 1 is not prepended for drawdown, so initial losses can be missed.
cum     = np.cumprod(1 + factor_rets)
max_dd  = np.max(1 - cum / np.maximum.accumulate(cum))
# Ordinary two-sided mean-zero t-test; no HAC adjustment or multiple-test correction.
t_stat, p_val = stats.ttest_1samp(factor_rets, 0)

print(f'{"Metric":<25} {"Ours (VW, 84 stks)":>20} {"Paper (VW, 357 stks)":>22}')
print('-'*70)
print(f'{"Ann. Return":<25} {ann_ret:>19.2%} {"2.21%":>22}')
print(f'{"Ann. Volatility":<25} {ann_vol:>19.2%} {"~10-13%":>22}')
print(f'{"Sharpe Ratio":<25} {ann_sr:>19.3f} {"~0.27":>22}')
print(f'{"Max Drawdown":<25} {max_dd:>19.2%} {"not reported":>22}')
print(f'{"t-stat":<25} {t_stat:>19.3f}')
print(f'{"p-value":<25} {p_val:>19.3f}')

# CAPM alpha
spy_ser  = spy_ex.copy()
fac_pidx = pd.PeriodIndex(factor_dates, freq='M')
mkt_arr  = spy_ser.reindex(fac_pidx).values
valid    = ~np.isnan(mkt_arr)
s, ic, _, _, _ = stats.linregress(mkt_arr[valid], factor_rets[valid])
print(f'{"CAPM alpha (ann.)":<25} {ic*12:>19.2%}   beta: {s:.3f}')
print(f'\nPaper: CAPM alpha 2.21% (t=1.09) — peripheral stocks outperform on VW basis')

## 5. Experiment 3: market-return prediction

* Match the change in mean centrality to the SPY excess-return approximation for the prediction month. Network fits underlying that signal end before the prediction month. Missing pairs are removed before regression.

* Fit an intercept and slope over the full paired sample, then produce expanding-window predictions after `BURN_IN` observations. The comparator predicts the expanding historical mean. Out-of-sample R² compares their squared errors; positive values indicate lower squared error for the signal model over this sample.

* The reported robust t-statistic uses **HC0 heteroskedasticity-only covariance**, not Newey–West/HAC: the calculation contains no lagged score cross-products.

* For the certainty-equivalent comparison, both strategies use time-varying weights based on expected return, historical variance and risk aversion, clipped to −0.5 through 1.5. The comparator is not constant-weight. The reported gain is annualized mean-variance utility on the excess-return approximation, with no trading costs. A negative fitted slope alone does not establish crash prediction or successful replication.


In [ ]:
mkt_excess = spy_ex.reindex(dens_dates).values
valid = ~np.isnan(delta_d) & ~np.isnan(mkt_excess)
X_reg, Y_reg = delta_d[valid], mkt_excess[valid]
T_reg = valid.sum()
print(f'OBS: {T_reg} months')

# In-sample OLS
X_ols = np.column_stack([np.ones(T_reg), X_reg])
beta, _, _, _ = np.linalg.lstsq(X_ols, Y_reg, rcond=None)
a_is, b_is = beta
y_hat = X_ols @ beta
r2_is = 1 - np.sum((Y_reg - y_hat)**2) / np.sum((Y_reg - Y_reg.mean())**2)

# HC0 heteroskedasticity-only t-statistic (no serial-correlation correction)
resid   = Y_reg - y_hat
bread   = np.linalg.inv(X_ols.T @ X_ols)
score   = X_ols * resid[:, None]
# Sandwich covariance uses contemporaneous squared residual scores only.
# No Newey-West lag kernel or autocovariance terms appear here.
hc_var  = bread @ (score.T @ score) @ bread
t_beta  = b_is / np.sqrt(np.diag(hc_var)[1])

# Out-of-sample R² (Campbell-Thompson 2008)
oos_len    = T_reg - BURN_IN
y_hat_oos  = np.zeros(oos_len)
y_bar_oos  = np.zeros(oos_len)
# Refit on paired observations strictly before t_cur, then predict t_cur.
# The historical-mean forecast uses the same available observations.
for i in range(oos_len):
    t_cur = BURN_IN + i
    Xtr   = np.column_stack([np.ones(t_cur), X_reg[:t_cur]])
    b, _, _, _ = np.linalg.lstsq(Xtr, Y_reg[:t_cur], rcond=None)
    y_hat_oos[i] = b[0] + b[1] * X_reg[t_cur]
    y_bar_oos[i] = Y_reg[:t_cur].mean()

Y_oos  = Y_reg[BURN_IN:]
r2_oos = (1 - np.sum((Y_oos - y_hat_oos)**2) / np.sum((Y_oos - y_bar_oos)**2)) * 100

# CER gain
# Estimate expanding sample variance from past outcomes and floor it for stability.
# Both signal and historical-mean forecasts lead to time-varying allocations.
sig2   = np.array([np.var(Y_reg[:BURN_IN+i], ddof=1) for i in range(oos_len)])
sig2   = np.maximum(sig2, 1e-6)
# Clip exposure between -50% short and 150% long. Returns below are excess
# return approximations; financing, turnover and transaction costs are not modeled.
wm     = np.clip(y_hat_oos / (GAMMA * sig2), -0.5, 1.5)
wn     = np.clip(y_bar_oos / (GAMMA * sig2), -0.5, 1.5)
rm, rn = wm * Y_oos, wn * Y_oos
cer_gain = ((rm.mean() - GAMMA/2 * rm.var(ddof=1)) - (rn.mean() - GAMMA/2 * rn.var(ddof=1))) * 12 * 100

print(f'\n{"Metric":<30} {"Ours":>10} {"Paper (bivariate)":>18}')
print('-'*60)
print(f'{"β (sign)":<30} {b_is:>10.4f} {"< 0 (negative)":>18}')
print(f'{"t(β) robust":<30} {t_beta:>10.2f} {"−1.62":>18}')
print(f'{"In-sample R²":<30} {r2_is*100:>9.2f}% {"0.75%":>18}')
print(f'{"Out-of-sample R²_OS":<30} {r2_oos:>9.2f}% {"0.55%":>18}')
print(f'{"CER gain Δ (ann.)":<30} {cer_gain:>9.2f}% {"1.83%":>18}')
print(f'{"β < 0 confirmed?":<30} {"Yes ✓" if b_is < 0 else "No":>10}')
print(f'\nPaper: dense network → lower expected returns (β<0, network density predicts crashes)')
print(f'Paper multivariate (with 21 controls): R²=5.98%, R²_OS=3.40%, CER gain=8.16%')


## 6. Diagnostic charts

- **A:** Compounded growth of the approximate size-weighted log-return spread.
- **B:** Mean L2-normalized eigenvector centrality, with manually specified recession shading. Dates mark prediction months, rather than the last month included in each fitted network.
- **C:** Paired centrality changes and market returns, with the in-sample regression line and HC0 t-statistic.
- **D:** Cumulative signal-model squared error divided by cumulative historical-mean squared error. Below one means lower error so far.

The charts visualize the implementation above; they do not resolve its return conventions, preprocessing or historical-size limitations.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Howard, Lohre & Mudde (2025) Replication\n'
             'Exp 2 (Low Centrality Factor) & Exp 3 (Market Timing)\n'
             '84 S&P 500 stocks | Lasso network (λ=0.1) | 48-month window | 2000–2022',
             fontsize=11, fontweight='bold')

# Panel A: Low centrality factor cumulative return
ax = axes[0, 0]
cum = np.cumprod(1 + factor_rets)
fts = pd.PeriodIndex(factor_dates, freq='M').to_timestamp()
ax.plot(fts, cum, '#2196F3', lw=1.8, label=f'Low Centrality (size proxy) SR={ann_sr:.2f}')
ax.axhline(1, color='black', lw=0.5)
ax.fill_between(fts, 1, cum, where=cum >= 1, alpha=0.15, color='green')
ax.fill_between(fts, 1, cum, where=cum < 1, alpha=0.15, color='red')
ax.set_title('Exp 2: Low Centrality Factor\n(Long peripheral, Short central)', fontweight='bold')
ax.set_ylabel('Cumulative Wealth ($1)'); ax.legend(); ax.grid(True, alpha=0.3)

# Panel B: Mean normalized centrality; this is not literal edge density.
ax = axes[0, 1]
pts = pd.PeriodIndex(pred_dates, freq='M').to_timestamp()
ax.plot(pts, avg_cent, '#FF9800', lw=1.5)
ax.axhline(avg_cent.mean(), color='black', lw=0.8, ls='--', label='Mean')
for rs, re in [('2001-03','2001-11'),('2007-12','2009-06'),('2020-02','2020-04')]:
    ax.axvspan(pd.Timestamp(rs), pd.Timestamp(re), alpha=0.15, color='gray')
ax.set_title('Average Network Centrality (Density Proxy)\n'
             'Shaded: NBER recessions', fontweight='bold')
ax.set_ylabel('Mean Eigenvector Centrality (L2 norm)'); ax.legend(); ax.grid(True, alpha=0.3)

# Panel C: Scatter Δd vs next-month S&P 500 return
ax = axes[1, 0]
ax.scatter(X_reg, Y_reg, alpha=0.25, s=12, color='#9C27B0')
xl = np.linspace(X_reg.min(), X_reg.max(), 100)
ax.plot(xl, a_is + b_is * xl, 'red', lw=2,
        label=f'β={b_is:.2f} (t={t_beta:.2f})\nR²={r2_is*100:.2f}%')
ax.axhline(0, color='black', lw=0.5); ax.axvline(0, color='black', lw=0.5)
ax.set_title('Exp 3: Network Density Change vs Market Return\n'
             r'$r_{t+1} = \alpha + \beta\cdot\Delta d_t + \varepsilon$', fontweight='bold')
ax.set_xlabel('Δ Mean Centrality'); ax.set_ylabel('S&P 500 Excess Return (monthly)')
ax.legend(); ax.grid(True, alpha=0.3)

# Panel D: OOS cumulative MSE ratio
ax = axes[1, 1]
# At every point compare cumulative squared errors on the identical OOS sample.
# The final value equals 1 - r2_oos/100; an early ratio can be unstable.
ratio = np.cumsum((Y_oos - y_hat_oos)**2) / np.cumsum((Y_oos - y_bar_oos)**2)
ax.plot(range(len(ratio)), ratio, '#F44336', lw=1.5)
ax.axhline(1.0, color='black', lw=1, ls='--', label='Break-even')
ax.fill_between(range(len(ratio)), 1, ratio, where=ratio < 1,
                alpha=0.2, color='green', label=f'Model wins\n(OOS R²={r2_oos:.2f}%)')
ax.fill_between(range(len(ratio)), 1, ratio, where=ratio >= 1, alpha=0.2, color='red')
ax.set_title(f'Exp 3: Cumulative OOS MSE Ratio\nR²_OS={r2_oos:.2f}%  CER gain={cer_gain:.2f}%',
             fontweight='bold')
ax.set_xlabel(f'OOS months (burn-in={BURN_IN})')
ax.set_ylabel('MSE ratio (< 1 = model beats mean)'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
